In [168]:
# 0. import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
%matplotlib inline
from matplotlib.ticker import MaxNLocator
import matplotlib.dates as mdates
import datetime
import json
import statsmodels.api as sm
from patsy import dmatrix
from pathlib import Path


In [169]:
# read data — paths do not depend on os.getcwd()
PROJECT_ROOT = Path("/Users/xueqingliu/Harvard University Dropbox/Liu Xueqing/ADAPR-MRT-Testbed")
COMBINED_DIR = Path("/Users/xueqingliu/Harvard University Dropbox/Liu Xueqing/ADAPT_MRT/rawdata/_combined")
WORK_DIR = PROJECT_ROOT / "env_para_vanilla"
WORK_DIR.mkdir(parents=True, exist_ok=True)

df_merged = pd.read_csv(COMBINED_DIR / "df_merged.csv")

print("COMBINED_DIR:", COMBINED_DIR.resolve())
print("WORK_DIR:   ", WORK_DIR.resolve(), "| exists:", WORK_DIR.is_dir())

# Aliases for later cells that use `folder` / `work_folder` (Path so `/` joins work)
folder = COMBINED_DIR
work_folder = Path(WORK_DIR)

COMBINED_DIR: /Users/xueqingliu/Harvard University Dropbox/Liu Xueqing/ADAPT_MRT/rawdata/_combined
WORK_DIR:    /Users/xueqingliu/Harvard University Dropbox/Liu Xueqing/ADAPR-MRT-Testbed/env_para_vanilla | exists: True


In [170]:
# display all columns
print(df_merged.columns)

Index(['ParticipantIdentifier', 'Date', 'DecisionTime', 'DateTimeStart',
       '4hour_step', 'CheckStatus_x', 'EMA_StepCount', 'YesterdayStepCount',
       'prior2hour_step', 'CheckStatus_y', 'RecordedPhysicalActivity',
       'Previous7DaysRPA', 'DayWearing', 'ValidHours', 'past7days_daywearing',
       'nextday_wearing', 'restday_valid_minutes', 'morning_wearing',
       'restday_wearing', 'DailyPageviewCount', 'YesterdayPageviewCount',
       'Past7DaysPageviewEMA', 'TomorrowPageviewCount', 'DatetimeStart',
       'HourlyPageviewCount', 'week', 'year', 'week_present',
       'AffectiveValuation', 'CAE-1', 'CAE-10', 'CAE-11', 'CAE-12', 'CAE-2',
       'CAE-3', 'CAE-4', 'CAE-5', 'CAE-6', 'CAE-7', 'CAE-8', 'CAE-9',
       'Exp-tool-1', 'Exp-tool-2', 'week_present_lastweek',
       'AffectiveValuation_lastweek', 'CAE-1_lastweek', 'CAE-10_lastweek',
       'CAE-11_lastweek', 'CAE-12_lastweek', 'CAE-2_lastweek',
       'CAE-3_lastweek', 'CAE-4_lastweek', 'CAE-5_lastweek', 'CAE-6_lastweek

In [171]:
# select columns
df_fit = df_merged[['ParticipantIdentifier', 'Date', 'DecisionTime', 'week', 'day', 'dow', 'is_weekend',
                    'WalkingSuggestion', 'Interacted_walk', 'Interacted_7d_walk',
                    'SalienceMessage', 'Interacted_salience', 'Interacted_7d_salience', 
                    'yesterday_SalienceMessage', 'yesterday_Interacted', 'yesterday_Interacted_7d',
                    'planning_prompt', 'yesterday_planning_prompt',
                    '4hour_step', 'EMA_StepCount', 'YesterdayStepCount', 'prior2hour_step', 'RecordedPhysicalActivity',
                    'Previous7DaysRPA', 'DayWearing', 'nextday_wearing', 'past7days_daywearing', 'morning_wearing',
                    'DailyPageviewCount', 'Past7DaysPageviewEMA', 'TomorrowPageviewCount', 'HourlyPageviewCount',
                    'week_present', 'week_present_lastweek', 'daily_present', 'daily_present_yesterday',
                    'affective_reflection', 'anticipated_affect', 'affective_reflection_yesterday', 'anticipated_affect_yesterday',
                    'CAE_avg', 'CAE_avg_lastweek', 'CAE_short_avg', 'recent_burden',
                    'Exp-tool-1', 'Exp-tool-2', 'Exp-tool-1_lastweek', 'Exp-tool-2_lastweek'
                  ]]

print(df_fit)

      ParticipantIdentifier        Date  DecisionTime  week  day  dow  \
0                       118  2025-09-14             0     0    0    7   
1                       118  2025-09-14             1     0    0    7   
2                       118  2025-09-15             0     0    1    1   
3                       118  2025-09-15             1     0    1    1   
4                       118  2025-09-16             0     0    2    2   
...                     ...         ...           ...   ...  ...  ...   
1865                    225  2026-03-07             1    11   82    6   
1866                    225  2026-03-08             0    11   83    7   
1867                    225  2026-03-08             1    11   83    7   
1868                    225  2026-03-09             0    12   84    1   
1869                    225  2026-03-09             1    12   84    1   

      is_weekend  WalkingSuggestion  Interacted_walk  Interacted_7d_walk  ...  \
0              1                0.0       

In [172]:
# normalize day and week to be within [-1,1] 
# TODO: based on the expected range of RCT
# TODO: I'll use the MRT range for now, 84 and 12
df_fit = df_fit.copy()
df_fit['day_norm'] = (df_fit['day'] - (1+84)/2) / ((84-1)/2)
df_fit['week_norm'] = (df_fit['week'] - (1+12)/2) / ((12-1)/2)
df_fit['dow_norm'] = (df_fit['dow'] - (1+7)/2) / ((7-1)/2)

print(df_fit)

# remove the first day of data for each participant


      ParticipantIdentifier        Date  DecisionTime  week  day  dow  \
0                       118  2025-09-14             0     0    0    7   
1                       118  2025-09-14             1     0    0    7   
2                       118  2025-09-15             0     0    1    1   
3                       118  2025-09-15             1     0    1    1   
4                       118  2025-09-16             0     0    2    2   
...                     ...         ...           ...   ...  ...  ...   
1865                    225  2026-03-07             1    11   82    6   
1866                    225  2026-03-08             0    11   83    7   
1867                    225  2026-03-08             1    11   83    7   
1868                    225  2026-03-09             0    12   84    1   
1869                    225  2026-03-09             1    12   84    1   

      is_weekend  WalkingSuggestion  Interacted_walk  Interacted_7d_walk  ...  \
0              1                0.0       

In [173]:
# log transform the step count data
df_fit['4hour_step'] = np.log(df_fit['4hour_step'] + 1)
df_fit['YesterdayStepCount'] = np.log(df_fit['YesterdayStepCount'] + 1)
df_fit['prior2hour_step'] = np.log(df_fit['prior2hour_step'] + 1)

In [174]:
# standardize the rest of the data
digits = 3
df_fit = df_fit.copy()
step_count_shift = np.round(np.mean(df_fit['4hour_step']), digits)
step_count_scale = np.round(np.std(df_fit['4hour_step']), digits)
df_fit['4hour_step_norm'] = (df_fit['4hour_step'] - step_count_shift) / step_count_scale

yesterday_step_count_shift = np.round(np.mean(df_fit['YesterdayStepCount']), digits)
yesterday_step_count_scale = np.round(np.std(df_fit['YesterdayStepCount']), digits)
df_fit['YesterdayStepCount_norm'] = (df_fit['YesterdayStepCount'] - yesterday_step_count_shift) / yesterday_step_count_scale

prior30_step_count_shift = np.round(np.mean(df_fit['prior2hour_step']), digits)
prior30_step_count_scale = np.round(np.std(df_fit['prior2hour_step']), digits)
df_fit['prior2hour_step_norm'] = (df_fit['prior2hour_step'] - prior30_step_count_shift) / prior30_step_count_scale

EMA_StepCount_shift = np.round(np.mean(df_fit['EMA_StepCount']), digits)
EMA_StepCount_scale = np.round(np.std(df_fit['EMA_StepCount']), digits)
df_fit['EMA_StepCount_norm'] = (df_fit['EMA_StepCount'] - EMA_StepCount_shift) / EMA_StepCount_scale

DailyPageviewCount_shift = np.round(np.mean(df_fit['DailyPageviewCount']), digits)
DailyPageviewCount_scale = np.round(np.std(df_fit['DailyPageviewCount']), digits)
df_fit['DailyPageviewCount_norm'] = (df_fit['DailyPageviewCount'] - DailyPageviewCount_shift) / DailyPageviewCount_scale

Past7DaysPageviewEMA_shift = np.round(np.mean(df_fit['Past7DaysPageviewEMA']), digits)
Past7DaysPageviewEMA_scale = np.round(np.std(df_fit['Past7DaysPageviewEMA']), digits)
df_fit['Past7DaysPageviewEMA_norm'] = (df_fit['Past7DaysPageviewEMA'] - Past7DaysPageviewEMA_shift) / Past7DaysPageviewEMA_scale

TomorrowPageviewCount_shift = np.round(np.mean(df_fit['TomorrowPageviewCount']), digits)
TomorrowPageviewCount_scale = np.round(np.std(df_fit['TomorrowPageviewCount']), digits)
df_fit['TomorrowPageviewCount_norm'] = (df_fit['TomorrowPageviewCount'] - TomorrowPageviewCount_shift) / TomorrowPageviewCount_scale

HourlyPageviewCount_shift = np.round(np.mean(df_fit['HourlyPageviewCount']), digits)
HourlyPageviewCount_scale = np.round(np.std(df_fit['HourlyPageviewCount']), digits)
df_fit['HourlyPageviewCount_norm'] = (df_fit['HourlyPageviewCount'] - HourlyPageviewCount_shift) / HourlyPageviewCount_scale

affective_reflection_shift = np.round(np.mean(df_fit['affective_reflection']), digits)
affective_reflection_scale = np.round(np.std(df_fit['affective_reflection']), digits)
df_fit['affective_reflection_norm'] = (df_fit['affective_reflection'] - affective_reflection_shift) / affective_reflection_scale

anticipated_affect_shift = np.round(np.mean(df_fit['anticipated_affect']), digits)
anticipated_affect_scale = np.round(np.std(df_fit['anticipated_affect']), digits)
df_fit['anticipated_affect_norm'] = (df_fit['anticipated_affect'] - anticipated_affect_shift) / anticipated_affect_scale

affective_reflection_yesterday_shift = np.round(np.mean(df_fit['affective_reflection_yesterday']), digits)
affective_reflection_yesterday_scale = np.round(np.std(df_fit['affective_reflection_yesterday']), digits)
df_fit['affective_reflection_yesterday_norm'] = (df_fit['affective_reflection_yesterday'] - affective_reflection_yesterday_shift) / affective_reflection_yesterday_scale

anticipated_affect_yesterday_shift = np.round(np.mean(df_fit['anticipated_affect_yesterday']), digits)
anticipated_affect_yesterday_scale = np.round(np.std(df_fit['anticipated_affect_yesterday']), digits)
df_fit['anticipated_affect_yesterday_norm'] = (df_fit['anticipated_affect_yesterday'] - anticipated_affect_yesterday_shift) / anticipated_affect_yesterday_scale

CAE_avg_shift = np.round(np.mean(df_fit['CAE_avg']), digits)
CAE_avg_scale = np.round(np.std(df_fit['CAE_avg']), digits)
df_fit['CAE_avg_norm'] = (df_fit['CAE_avg'] - CAE_avg_shift) / CAE_avg_scale

CAE_avg_lastweek_shift = np.round(np.mean(df_fit['CAE_avg_lastweek']), digits)
CAE_avg_lastweek_scale = np.round(np.std(df_fit['CAE_avg_lastweek']), digits)
df_fit['CAE_avg_lastweek_norm'] = (df_fit['CAE_avg_lastweek'] - CAE_avg_lastweek_shift) / CAE_avg_lastweek_scale

CAE_short_avg_shift = np.round(np.mean(df_fit['CAE_short_avg']), digits)
CAE_short_avg_scale = np.round(np.std(df_fit['CAE_short_avg']), digits)
df_fit['CAE_short_avg_norm'] = (df_fit['CAE_short_avg'] - CAE_short_avg_shift) / CAE_short_avg_scale


recent_burden_shift = np.round(np.mean(df_fit['recent_burden']), digits)
recent_burden_scale = np.round(np.std(df_fit['recent_burden']), digits)
df_fit['recent_burden_norm'] = (df_fit['recent_burden'] - recent_burden_shift) / recent_burden_scale

# perceived utility need to be normalized later because we will tune it (what is true vs what we use in the algorithm)


# check the range of the normalized data
hour4_step_count_limit = [np.round(np.min(df_fit['4hour_step_norm']), digits), np.round(np.max(df_fit['4hour_step_norm']), digits)]
yesterday_step_count_limit = [np.round(np.min(df_fit['YesterdayStepCount_norm']), digits), np.round(np.max(df_fit['YesterdayStepCount_norm']), digits)]
prior2hour_step_count_limit = [np.round(np.min(df_fit['prior2hour_step_norm']), digits), np.round(np.max(df_fit['prior2hour_step_norm']), digits)]
EMA_step_count_limit = [np.round(np.min(df_fit['EMA_StepCount_norm']), digits), np.round(np.max(df_fit['EMA_StepCount_norm']), digits)]
daily_pageview_count_limit = [np.round(np.min(df_fit['DailyPageviewCount_norm']), digits), np.round(np.max(df_fit['DailyPageviewCount_norm']), digits)]
past7days_pageview_count_limit = [np.round(np.min(df_fit['Past7DaysPageviewEMA_norm']), digits), np.round(np.max(df_fit['Past7DaysPageviewEMA_norm']), digits)]
tomorrow_pageview_count_limit = [np.round(np.min(df_fit['TomorrowPageviewCount_norm']), digits), np.round(np.max(df_fit['TomorrowPageviewCount_norm']), digits)]
hourly_pageview_count_limit = [np.round(np.min(df_fit['HourlyPageviewCount_norm']), digits), np.round(np.max(df_fit['HourlyPageviewCount_norm']), digits)]
affective_reflection_limit = [np.round(np.min(df_fit['affective_reflection_norm']), digits), np.round(np.max(df_fit['affective_reflection_norm']), digits)]
anticipated_affect_limit = [np.round(np.min(df_fit['anticipated_affect_norm']), digits), np.round(np.max(df_fit['anticipated_affect_norm']), digits)]
affective_reflection_yesterday_limit = [np.round(np.min(df_fit['affective_reflection_yesterday_norm']), digits), np.round(np.max(df_fit['affective_reflection_yesterday_norm']), digits)]
anticipated_affect_yesterday_limit = [np.round(np.min(df_fit['anticipated_affect_yesterday_norm']), digits), np.round(np.max(df_fit['anticipated_affect_yesterday_norm']), digits)]
CAE_avg_limit = [np.round(np.min(df_fit['CAE_avg_norm']), digits), np.round(np.max(df_fit['CAE_avg_norm']), digits)]
CAE_avg_lastweek_limit = [np.round(np.min(df_fit['CAE_avg_lastweek_norm']), digits), np.round(np.max(df_fit['CAE_avg_lastweek_norm']), digits)]
CAE_short_avg_limit = [np.round(np.min(df_fit['CAE_short_avg_norm']), digits), np.round(np.max(df_fit['CAE_short_avg_norm']), digits)]
recent_burden_limit = [np.round(np.min(df_fit['recent_burden_norm']), digits), np.round(np.max(df_fit['recent_burden_norm']), digits)]


# save the shift and scales into a json file
std_params = {
    '4hour_step_count_shift': step_count_shift,
    '4hour_step_count_scale': step_count_scale,
    'yesterday_step_count_shift': yesterday_step_count_shift,
    'yesterday_step_count_scale': yesterday_step_count_scale,
    'prior2hour_step_count_shift': prior30_step_count_shift,
    'prior2hour_step_count_scale': prior30_step_count_scale,
    'EMA_step_count_shift': EMA_StepCount_shift,
    'EMA_step_count_scale': EMA_StepCount_scale,
    'DailyPageviewCount_shift': DailyPageviewCount_shift,
    'DailyPageviewCount_scale': DailyPageviewCount_scale,
    'Past7DaysPageviewEMA_shift': Past7DaysPageviewEMA_shift,
    'Past7DaysPageviewEMA_scale': Past7DaysPageviewEMA_scale,
    'TomorrowPageviewCount_shift': TomorrowPageviewCount_shift,
    'TomorrowPageviewCount_scale': TomorrowPageviewCount_scale,
    'HourlyPageviewCount_shift': HourlyPageviewCount_shift,
    'HourlyPageviewCount_scale': HourlyPageviewCount_scale,
    'affective_reflection_shift': affective_reflection_shift,
    'affective_reflection_scale': affective_reflection_scale,
    'anticipated_affect_shift': anticipated_affect_shift,
    'anticipated_affect_scale': anticipated_affect_scale,
    'affective_reflection_yesterday_shift': affective_reflection_yesterday_shift,
    'affective_reflection_yesterday_scale': affective_reflection_yesterday_scale,
    'anticipated_affect_yesterday_shift': anticipated_affect_yesterday_shift,
    'anticipated_affect_yesterday_scale': anticipated_affect_yesterday_scale,
    'CAE_avg_shift': CAE_avg_shift,
    'CAE_avg_scale': CAE_avg_scale,
    'CAE_avg_lastweek_shift': CAE_avg_lastweek_shift,
    'CAE_avg_lastweek_scale': CAE_avg_lastweek_scale,
    'CAE_short_avg_shift': CAE_short_avg_shift,
    'CAE_short_avg_scale': CAE_short_avg_scale,
    'recent_burden_shift': recent_burden_shift,
    'recent_burden_scale': recent_burden_scale,
    '4hour_step_count_limit': hour4_step_count_limit,
    'YesterdayStepCount_limit': yesterday_step_count_limit,
    'prior2hour_step_count_limit': prior2hour_step_count_limit,
    'EMA_StepCount_limit': EMA_step_count_limit,
    'DailyPageviewCount_limit': daily_pageview_count_limit,
    'Past7DaysPageviewEMA_limit': past7days_pageview_count_limit,
    'TomorrowPageviewCount_limit': tomorrow_pageview_count_limit,
    'HourlyPageviewCount_limit': hourly_pageview_count_limit,
    'affective_reflection_limit': affective_reflection_limit,
    'anticipated_affect_limit': anticipated_affect_limit,
    'affective_reflection_yesterday_limit': affective_reflection_yesterday_limit,
    'anticipated_affect_yesterday_limit': anticipated_affect_yesterday_limit,
    'CAE_avg_limit': CAE_avg_limit,
    'CAE_avg_lastweek_limit': CAE_avg_lastweek_limit,
    'CAE_short_avg_limit': CAE_short_avg_limit,
    'recent_burden_limit': recent_burden_limit
}

_work_dir = Path(work_folder)
output_path = _work_dir / "std_params.json"
os.makedirs(_work_dir, exist_ok=True)
with open(output_path, 'w') as f:
    json.dump(std_params, f)





In [175]:
# only save the normalized data
# remove the unnormalized data
# df_fit = df_fit.drop(columns=['day', 'week',
#                               'step_count', 'yesterday_step_count', 'prior30_step_count',
#                               'yesterday_pageview_count', 
#                               'AA_avg', 'perceived_utility',
#                               'affective_valuation',
#                               'perceived_utility_lastweek', 'AA_avg_lastweek',
#                               '7day_step_count_avg', '7day_step_count_std',
#                               'week_step_count_avg', 'week_walking_suggestion', 'week_view_status'])


df_fit.to_csv(folder / 'df_fit.csv', index=False)


In [176]:
print(df_fit.loc[df_fit['ParticipantIdentifier'] == 219, '4hour_step_norm'])

Series([], Name: 4hour_step_norm, dtype: float64)
